In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Étapes du transpileur

<details>
<summary><b>Versions des packages</b></summary>

Le code de cette page a été développé avec les dépendances suivantes.
Nous recommandons d'utiliser ces versions ou des versions plus récentes.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>

Cette page décrit les étapes du pipeline de transpilation préconstruit du SDK Qiskit. Il y a six étapes :

1. `init`
2. `layout`
3. `routing`
4. `translation`
5. `optimization`
6. `scheduling`

La fonction [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) crée un [gestionnaire de passes par étapes](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.StagedPassManager) préconfiguré, composé de ces étapes. Les passes spécifiques qui constituent chaque étape dépendent des arguments transmis à `generate_preset_pass_manager`. `optimization_level` est un argument positionnel obligatoire ; c'est un entier qui peut valoir 0, 1, 2 ou 3. Des valeurs plus élevées indiquent une optimisation plus poussée, mais aussi plus coûteuse en calcul (voir [Paramètres par défaut et options de configuration du transpileur](defaults-and-configuration-options)).

La méthode recommandée pour transpiler un circuit consiste à créer un gestionnaire de passes par étapes préconfiguré, puis à l'exécuter sur le circuit, comme décrit dans [Transpiler avec des gestionnaires de passes](transpile-with-pass-managers). Il existe toutefois une alternative plus simple, mais moins personnalisable : la fonction [`transpile`](https://docs.quantum.ibm.com/api/qiskit/compiler#qiskit.compiler.transpile). Cette fonction accepte directement le circuit en argument. Comme pour `generate_preset_pass_manager`, les passes de transpilation utilisées dépendent des arguments, tels que `optimization_level`, transmis à `transpile`. En interne, la fonction `transpile` appelle `generate_preset_pass_manager` pour créer un gestionnaire de passes préconfiguré et l'exécute sur le circuit.

## Étape init
Cette première étape fait très peu de choses par défaut et s'avère surtout utile si tu souhaites inclure tes propres optimisations initiales. Étant donné que la plupart des algorithmes de placement et de routage sont conçus pour fonctionner uniquement avec des portes à un ou deux qubits, cette étape sert aussi à traduire toute porte opérant sur plus de deux qubits en portes n'opérant que sur un ou deux qubits.

Pour plus d'informations sur la façon d'implémenter tes propres optimisations initiales pour cette étape, consulte la section sur les plugins et la personnalisation des gestionnaires de passes.

## Étape layout
L'étape suivante concerne le placement ou la connectivité du backend vers lequel un circuit sera envoyé. En général, les circuits quantiques sont des entités abstraites dont les qubits sont des représentations « virtuelles » ou « logiques » des qubits réels utilisés dans les calculs. Pour exécuter une séquence de portes, il est nécessaire d'établir une correspondance biunivoque entre les qubits « virtuels » et les qubits « physiques » d'un appareil quantique réel. Cette correspondance est stockée dans un objet `Layout` et fait partie des contraintes définies dans l'[architecture d'ensemble d'instructions (ISA)](./transpile#instruction-set-architecture) d'un backend.

![Cette image illustre la correspondance des qubits depuis la représentation en fils jusqu'à un diagramme représentant la façon dont les qubits sont connectés sur le QPU.](../docs/images/guides/transpiler-stages/layout-mapping.avif "Correspondance des qubits")

Le choix de la correspondance est extrêmement important pour minimiser le nombre d'opérations SWAP nécessaires à la projection du circuit d'entrée sur la topologie de l'appareil, et pour s'assurer que les qubits les mieux calibrés sont utilisés. En raison de l'importance de cette étape, les gestionnaires de passes préconfigurés essaient plusieurs méthodes différentes pour trouver le meilleur placement. Cela implique généralement deux étapes : d'abord, tenter de trouver un placement « parfait » (un placement qui ne nécessite aucune opération SWAP), puis exécuter une passe heuristique qui tente de trouver le meilleur placement si un placement parfait est impossible. Deux `Passes` sont généralement utilisées pour cette première étape :

- `TrivialLayout` : Fait correspondre naïvement chaque qubit virtuel au qubit physique de même numéro sur l'appareil (p. ex. [`0`,`1`,`2`,`3`] -> [`0`,`1`,`2`,`3`]). Il s'agit d'un comportement historique utilisé uniquement dans `optimization_level=1` pour tenter de trouver un placement parfait. En cas d'échec, `VF2Layout` est essayé ensuite.
- `VF2Layout` : Il s'agit d'une `AnalysisPass` qui sélectionne un placement idéal en traitant cette étape comme un problème d'isomorphisme de sous-graphe, résolu par l'algorithme VF2++. Si plusieurs placements sont trouvés, une heuristique de score est exécutée pour sélectionner la correspondance avec l'erreur moyenne la plus faible.

Ensuite, pour l'étape heuristique, deux passes sont utilisées par défaut :

- `DenseLayout` : Trouve le sous-graphe de l'appareil présentant la plus grande connectivité et ayant le même nombre de qubits que le circuit (utilisé pour le niveau d'optimisation 1 lorsque des opérations de flux de contrôle, telles que `IfElseOp`, sont présentes dans le circuit).
- `SabreLayout` : Cette passe sélectionne un placement en partant d'un placement aléatoire initial et en exécutant répétitivement l'algorithme `SabreSwap`. Cette passe n'est utilisée qu'aux niveaux d'optimisation 1, 2 et 3 si aucun placement parfait n'est trouvé via la passe `VF2Layout`. Pour plus de détails sur cet algorithme, consulte l'article [arXiv:1809.02573.](https://arxiv.org/abs/1809.02573)

## Étape routing
Pour implémenter une porte à deux qubits entre des qubits qui ne sont pas directement connectés sur un appareil quantique, il faut insérer une ou plusieurs portes SWAP dans le circuit afin de déplacer les états de qubits jusqu'à ce qu'ils soient adjacents sur la carte de portes de l'appareil. Chaque porte SWAP représente une opération coûteuse et bruyante à effectuer. Ainsi, trouver le nombre minimal de portes SWAP nécessaires pour projeter un circuit sur un appareil donné est une étape cruciale du processus de transpilation. Pour des raisons d'efficacité, cette étape est généralement calculée en même temps que l'étape Layout par défaut, mais elles sont logiquement distinctes l'une de l'autre. L'étape *Layout* sélectionne les qubits matériels à utiliser, tandis que l'étape *Routing* insère le nombre approprié de portes SWAP afin d'exécuter les circuits avec le placement sélectionné.

Cependant, trouver la correspondance SWAP optimale est difficile. En fait, il s'agit d'un problème NP-difficile, et son calcul est donc prohibitivement coûteux pour tout sauf les plus petits appareils quantiques et circuits d'entrée. Pour contourner ce problème, Qiskit utilise un algorithme heuristique stochastique appelé `SabreSwap` pour calculer une correspondance SWAP bonne, mais pas nécessairement optimale. L'utilisation d'une méthode stochastique signifie que les circuits générés ne sont pas garantis d'être identiques d'une exécution à l'autre. En effet, l'exécution répétée d'un même circuit produit une distribution de profondeurs de circuit et de nombres de portes en sortie. C'est pour cette raison que de nombreux utilisateurs choisissent d'exécuter la fonction de routage (ou l'intégralité du `StagedPassManager`) plusieurs fois et de sélectionner les circuits de plus faible profondeur parmi la distribution des sorties.

Par exemple, prenons un circuit GHZ à 15 qubits exécuté 100 fois, en utilisant un `initial_layout` « mauvais » (déconnecté).

In [1]:
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit_ibm_runtime.fake_provider import FakeAuckland, FakeWashingtonV2
from qiskit.transpiler import generate_preset_pass_manager

backend = FakeAuckland()

ghz = QuantumCircuit(15)
ghz.h(0)
ghz.cx(0, range(1, 15))

depths = []
for seed in range(100):
    pass_manager = generate_preset_pass_manager(
        optimization_level=1,
        backend=backend,
        layout_method="trivial",  # Fixed layout mapped in circuit order
        seed_transpiler=seed,  # For reproducible results
    )
    depths.append(pass_manager.run(ghz).depth())

plt.figure(figsize=(8, 6))
plt.hist(depths, align="left", color="#AC557C")
plt.xlabel("Depth", fontsize=14)
plt.ylabel("Counts", fontsize=14)

Text(0, 0.5, 'Counts')

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/358cfb50-02fc-48f2-a4ec-657837e0c304-1.svg" alt="Output of the previous code cell" />

This wide distribution demonstrates how difficult it is for the SWAP mapper to compute the best mapping.  To gain some insight, let's look at both the circuit being executed as well as the qubits that were chosen on the hardware.

In [2]:
ghz.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/bb3b8c1f-69fd-4e0c-9b78-9ee67f3361bb-0.svg" alt="Output of the previous code cell" />

In [3]:
from qiskit.visualization import plot_circuit_layout

# Plot the hardware graph and indicate which hardware qubits were chosen to run the circuit
transpiled_circ = pass_manager.run(ghz)
plot_circuit_layout(transpiled_circ, backend)

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/e4cd49ef-5b3e-4ee0-82f8-c83e1f0ae755-0.svg" alt="Output of the previous code cell" />

As you can see, this circuit has to execute a two-qubit gate between qubits 0 and 14, which are very far apart on the connectivity graph.  Running this circuit thus requires inserting SWAP gates to execute all of the two-qubit gates using the `SabreSwap` pass.

Note also that the `SabreSwap` algorithm is different from the larger `SabreLayout` method in the previous stage.  By default, `SabreLayout` runs both layout and routing, and returns the transformed circuit.  This is done for a few particular technical reasons specified in the pass's [API reference page](../api/qiskit/qiskit.transpiler.passes.SabreLayout).

## Translation stage

When writing a quantum circuit, you are free to use any quantum gate (unitary operation) that you like, along with a collection of non-gate operations such as qubit measurement or reset instructions.  However, most quantum devices only natively support a handful of quantum gate and non-gate operations.  These native gates are part of the definition of a target's [ISA](./transpile#instruction-set-architecture) and this stage of the preset `PassManagers`  translates (or *unrolls*) the gates specified in a circuit to the native basis gates of a specified backend.  This is an important step, as it allows the circuit to be executed by the backend, but typically leads to an increase in the depth and number of gates.

Two special cases are especially important to highlight, and help illustrate what this stage does.

1. If a SWAP gate is not a native gate to the target backend, this requires three CNOT gates:

In [4]:
print("native gates:" + str(sorted(backend.operation_names)))
qc = QuantumCircuit(2)
qc.swap(0, 1)
qc.decompose().draw("mpl")

native gates:['cx', 'delay', 'for_loop', 'id', 'if_else', 'measure', 'reset', 'rz', 'switch_case', 'sx', 'x']


<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/31fcf8b6-4f3a-460e-90fd-446813bd4f28-1.svg" alt="Output of the previous code cell" />

![Sortie de la cellule de code précédente](../docs/images/guides/transpiler-stages/extracted-outputs/e4cd49ef-5b3e-4ee0-82f8-c83e1f0ae755-0.svg)

Comme tu peux le voir, ce circuit doit exécuter une porte à deux qubits entre les qubits 0 et 14, qui sont très éloignés sur le graphe de connectivité. L'exécution de ce circuit nécessite donc l'insertion de portes SWAP pour exécuter toutes les portes à deux qubits via la passe `SabreSwap`.

Note également que l'algorithme `SabreSwap` est différent de la méthode `SabreLayout` plus large de l'étape précédente. Par défaut, `SabreLayout` exécute à la fois le placement et le routage, et retourne le circuit transformé. Ceci est fait pour quelques raisons techniques particulières précisées dans la [page de référence API](../api/qiskit/qiskit.transpiler.passes.SabreLayout) de la passe.

## Étape translation
Lors de l'écriture d'un circuit quantique, tu es libre d'utiliser n'importe quelle porte quantique (opération unitaire) de ton choix, ainsi qu'un ensemble d'opérations non-porte telles que les instructions de mesure ou de réinitialisation de qubit. Cependant, la plupart des appareils quantiques ne prennent en charge nativement qu'un petit nombre de portes quantiques et d'opérations non-porte. Ces portes natives font partie de la définition de l'[ISA](./transpile#instruction-set-architecture) d'une cible, et cette étape des `PassManagers` préconfigurés traduit (ou *déroule*) les portes spécifiées dans un circuit vers les portes de base natives d'un backend spécifié. Il s'agit d'une étape importante, car elle permet l'exécution du circuit par le backend, mais entraîne généralement une augmentation de la profondeur et du nombre de portes.

Deux cas particuliers méritent d'être soulignés et illustrent bien ce que fait cette étape.

1. Si une porte SWAP n'est pas une porte native du backend cible, elle nécessite trois portes CNOT :

In [5]:
qc = QuantumCircuit(3)
qc.ccx(0, 1, 2)
qc.decompose().draw("mpl")

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/8a2c1913-3324-44b0-859e-d7fd348161c3-0.svg" alt="Output of the previous code cell" />

For every Toffoli gate in a quantum circuit, the hardware may execute up to six CNOT gates and a handful of single-qubit gates.  This example demonstrates that any algorithm making use of multiple Toffoli gates will end up as a circuit with large depth and will therefore be appreciably affected by noise.

## Optimization stage

This stage centers around decomposing quantum circuits into the basis gate set of the target device, and must fight against the increased depth from the layout and routing stages.  Fortunately, there are many routines for optimizing circuits by either combining or eliminating gates.  In some cases, these methods are so effective that the output circuits have lower depth than the inputs, even after layout and routing to the hardware topology.  In other cases, not much can be done, and the computation may be difficult to perform on noisy devices.  This stage is where the various optimization levels begin to differ.

- For `optimization_level=1`, this stage prepares [`Optimize1qGatesDecomposition`](../api/qiskit/qiskit.transpiler.passes.Optimize1qGatesDecomposition) and [`CXCancellation`](/docs/api/qiskit/1.4/qiskit.transpiler.passes.CXCancellation), which combine chains of single-qubit gates and cancel any back-to-back CNOT gates.
- For `optimization_level=2`, this stage uses the [`CommutativeCancellation`](../api/qiskit/qiskit.transpiler.passes.CommutativeCancellation) pass instead of `CXCancellation`, which removes redundant gates by exploiting commutation relations.
- For `optimization_level=3`, this stage prepares the following passes:
  - [`Collect2qBlocks`](../api/qiskit/qiskit.transpiler.passes.Collect2qBlocks)
  - [`ConsolidateBlocks`](../api/qiskit/qiskit.transpiler.passes.ConsolidateBlocks)
  - [`UnitarySynthesis`](../api/qiskit/qiskit.transpiler.passes.UnitarySynthesis)
  - [`Optimize1qGateDecomposition`](../api/qiskit/qiskit.transpiler.passes.Optimize1qGatesDecomposition)
  - [`CommutativeCancellation`](../api/qiskit/qiskit.transpiler.passes.CommutativeCancellation)


Additionally, this stage also executes a few final checks to make sure that all instructions in the circuit are composed of the basis gates available on the target backend.

The example below using a GHZ state demonstrates the effects of different optimization level settings on circuit depth and gate count.

<Admonition type="note">
  The transpilation output varies due to the stochastic SWAP mapper. Therefore, the numbers below will likely change each time you run the code.
</Admonition>

![15-qubit GHZ state](../docs/images/guides/transpiler-stages/transpiler-11.avif "15-qubit GHZ state before transpilation")

The following code constructs a 15-qubit GHZ state and compares the `optimization_levels` of transpilation in terms of resulting circuit depth, gate counts, and multi-qubit gate counts.

In [6]:
ghz = QuantumCircuit(15)
ghz.h(0)
ghz.cx(0, range(1, 15))

depths = []
gate_counts = []
multiqubit_gate_counts = []
levels = [str(x) for x in range(4)]
for level in range(4):
    pass_manager = generate_preset_pass_manager(
        optimization_level=level,
        backend=backend,
        seed_transpiler=1234,
    )
    circ = pass_manager.run(ghz)
    depths.append(circ.depth())
    gate_counts.append(sum(circ.count_ops().values()))
    multiqubit_gate_counts.append(circ.count_ops()["cx"])

fig, (ax1, ax2) = plt.subplots(2, 1)
ax1.bar(levels, depths, label="Depth")
ax1.set_xlabel("Optimization Level")
ax1.set_ylabel("Depth")
ax1.set_title("Output Circuit Depth")
ax2.bar(levels, gate_counts, label="Number of Circuit Operations")
ax2.bar(levels, multiqubit_gate_counts, label="Number of CX gates")
ax2.set_xlabel("Optimization Level")
ax2.set_ylabel("Number of gates")
ax2.legend()
ax2.set_title("Number of output circuit gates")
fig.tight_layout()
plt.show()

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/2507de9c-1b94-4d56-b5a6-0b2bb1719a80-0.svg" alt="Output of the previous code cell" />

![Sortie de la cellule de code précédente](../docs/images/guides/transpiler-stages/extracted-outputs/31fcf8b6-4f3a-460e-90fd-446813bd4f28-1.svg)

En tant que produit de trois portes CNOT, un SWAP est une opération coûteuse à effectuer sur des appareils quantiques bruités. Cependant, ces opérations sont généralement nécessaires pour intégrer un circuit dans les connectivités de portes limitées de nombreux appareils. Ainsi, minimiser le nombre de portes SWAP dans un circuit est un objectif primordial du processus de transpilation.

2. Une porte Toffoli, ou porte contrôlée-contrôlée-NON (`ccx`), est une porte à trois qubits. Étant donné que notre ensemble de portes de base n'inclut que des portes à un et deux qubits, cette opération doit être décomposée. Cependant, le coût est élevé :

In [7]:
ghz = QuantumCircuit(5)
ghz.h(0)
ghz.cx(0, range(1, 5))


# Use fake backend
backend = FakeWashingtonV2()

# Run with optimization level 3 and 'asap' scheduling pass
pass_manager = generate_preset_pass_manager(
    optimization_level=3,
    backend=backend,
    scheduling_method="asap",
    seed_transpiler=1234,
)


circ = pass_manager.run(ghz)
circ.draw(output="mpl", idle_wires=False)

<Image src="../docs/images/guides/transpiler-stages/extracted-outputs/4339deb5-4947-493b-8940-e68c91311631-0.svg" alt="Output of the previous code cell" />

![Sortie de la cellule de code précédente](../docs/images/guides/transpiler-stages/extracted-outputs/8a2c1913-3324-44b0-859e-d7fd348161c3-0.svg)

Pour chaque porte Toffoli dans un circuit quantique, le matériel peut exécuter jusqu'à six portes CNOT et quelques portes à un qubit. Cet exemple montre que tout algorithme utilisant plusieurs portes Toffoli aboutira à un circuit de grande profondeur et sera donc considérablement affecté par le bruit.

## Étape optimization
Cette étape est centrée sur la décomposition des circuits quantiques vers l'ensemble de portes de base de l'appareil cible, et doit lutter contre l'augmentation de profondeur due aux étapes de placement et de routage. Heureusement, il existe de nombreuses routines pour optimiser les circuits en combinant ou en éliminant des portes. Dans certains cas, ces méthodes sont si efficaces que les circuits en sortie ont une profondeur inférieure aux circuits en entrée, même après le placement et le routage vers la topologie matérielle. Dans d'autres cas, il n'y a pas grand-chose à faire et le calcul peut être difficile à réaliser sur des appareils bruités. C'est lors de cette étape que les différents niveaux d'optimisation commencent à se distinguer.

- Pour `optimization_level=1`, cette étape prépare [`Optimize1qGatesDecomposition`](../api/qiskit/qiskit.transpiler.passes.Optimize1qGatesDecomposition) et [`CXCancellation`](https://docs.quantum.ibm.com/api/qiskit/1.4/qiskit.transpiler.passes.CXCancellation), qui combinent des chaînes de portes à un qubit et annulent les portes CNOT dos-à-dos.
- Pour `optimization_level=2`, cette étape utilise la passe [`CommutativeCancellation`](../api/qiskit/qiskit.transpiler.passes.CommutativeCancellation) au lieu de `CXCancellation`, qui supprime les portes redondantes en exploitant les relations de commutation.
- Pour `optimization_level=3`, cette étape prépare les passes suivantes :
  - [`Collect2qBlocks`](../api/qiskit/qiskit.transpiler.passes.Collect2qBlocks)
  - [`ConsolidateBlocks`](../api/qiskit/qiskit.transpiler.passes.ConsolidateBlocks)
  - [`UnitarySynthesis`](../api/qiskit/qiskit.transpiler.passes.UnitarySynthesis)
  - [`Optimize1qGateDecomposition`](../api/qiskit/qiskit.transpiler.passes.Optimize1qGatesDecomposition)
  - [`CommutativeCancellation`](../api/qiskit/qiskit.transpiler.passes.CommutativeCancellation)

De plus, cette étape exécute également quelques vérifications finales pour s'assurer que toutes les instructions du circuit sont composées des portes de base disponibles sur le backend cible.

L'exemple ci-dessous utilisant un état GHZ illustre les effets des différents niveaux d'optimisation sur la profondeur du circuit et le nombre de portes.

> **Note:** La sortie de transpilation varie en raison du mappeur SWAP stochastique. Par conséquent, les valeurs ci-dessous changeront probablement à chaque exécution du code.

![État GHZ à 15 qubits](../docs/images/guides/transpiler-stages/transpiler-11.avif "État GHZ à 15 qubits avant transpilation")

Le code suivant construit un état GHZ à 15 qubits et compare les `optimization_levels` de transpilation en termes de profondeur de circuit résultante, de nombres de portes et de nombres de portes multi-qubits.